# 章节实践

通过本章的学习，我们已经掌握了MC2融合算子的核心概念和开发流程。为了巩固所学知识，现提供以下实践练习：

实现MC2融合算子MatmulAllGather，这是一个典型的矩阵乘法与集合通信AllGather融合的算子，广泛应用于分布式训练场景。

相关算法：  
<div style="width: 30%; float: left; clear: both; margin: 10px 0;">

$$
\text{MatmulAllGather}(Y) = \text{AllGather}\left(\text{Matmul}(A, B)\right)
$$
</div>
<div style="clear: both;"></div>

其中：
- Matmul: 矩阵乘法运算 $C = A \times B$，输出形状为 $(M, N)$
- AllGather: 集合通信操作，将所有rank的Matmul结果C汇聚到输出$Y$，最终输出形状为 $(rankDim \times M, N)$

**与ReduceScatter的区别**：
- ReduceScatter: 输出大小 = Matmul输出 / rankDim（切分）
- AllGather: 输出大小 = Matmul输出 × rankDim（汇聚）

算子原型：
<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulAllGather</td>
  </tr>
  <tr>
    <td rowspan="3" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">A</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(M, K)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bfloat16</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(K, N)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bfloat16</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">Y</td>
    <td style="padding: 8px;">(rankDim × M, N)</td>
    <td style="padding: 8px;">bfloat16</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

要求：

- 算子命名为MatmulAllGather。

- 完成MatmulAllGather算子host侧Tiling实现相关代码。

- 完成MatmulAllGather算子kernel侧核函数实现相关代码。

- 支持bfloat16类型输入输出。

- 支持2卡或4卡分布式场景，rankDim可配置。

- 支持 M = 1024, N = 1024, K = 1024 的输入。

- 实现流水线优化：Matmul结果直接写入Win Area，避免中间拷贝。

请开始你的实践，体验MC2融合算子的完整开发过程。

---

环境准备：

In [ ]:
!mkdir -p Sources/01.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

--- 
### **算子工程创建**

MC2融合算子采用CMake工程结构，适合多卡分布式场景。工程目录结构如下：

```
matmul_all_gather/
├── CMakeLists.txt          # CMake编译配置
├── build.sh                # 编译脚本
├── run_multi_data_test.sh  # 多卡测试脚本
├── include/                # 头文件
│   ├── tiling_data.h       # Tiling数据结构
│   ├── utils.h             # 工具函数
│   ├── kernel/             # Kernel实现头文件
│   │   ├── matmul_all_gather.h  # MatmulAllGather算子类
│   │   └── mte_comm.h      # MTE通信类
│   └── block/              # Block级别实现
│       ├── simple_block_mmad.h         # Matmul Block实现
│       ├── simple_block_scheduler.h    # Block调度器
│       └── block_scheduler_utils.h     # 调度工具
├── src/                    # 源代码
│   ├── matmul_all_gather.cpp        # Kernel入口
│   └── test_matmul_all_gather.cpp   # Host测试程序
└── scripts/                # 测试脚本
    └── verify_precision.py # 结果验证
```

In [ ]:
# 创建算子工程目录结构
!mkdir -p Sources/01.04/matmul_all_gather/include/kernel
!mkdir -p Sources/01.04/matmul_all_gather/include/block
!mkdir -p Sources/01.04/matmul_all_gather/src
!mkdir -p Sources/01.04/matmul_all_gather/scripts
print("\n✅ Project directory structure created successfully!")

接下来创建CMakeLists.txt编译配置文件：

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(matmul_all_gather)

set(CMAKE_EXPORT_COMPILE_COMMANDS ON)

if(NOT CMAKE_BUILD_TYPE)
    set(CMAKE_BUILD_TYPE "Debug" CACHE STRING "Build type Release/Debug" FORCE)
endif()

set(SOC_VERSION "ascend910b" CACHE STRING "system on chip type")

if(DEFINED ENV{ASCEND_HOME_PATH})
    set(ASCEND_CANN_PACKAGE_PATH $ENV{ASCEND_HOME_PATH})
else()
    message(FATAL_ERROR "ASCEND_HOME_PATH not set, please configure CANN environment first")
endif()

set(ASCENDC_CMAKE_DIR "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake")
if(NOT EXISTS "${ASCENDC_CMAKE_DIR}")
    set(ASCENDC_CMAKE_DIR "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake")
endif()
if(NOT EXISTS "${ASCENDC_CMAKE_DIR}")
    message(FATAL_ERROR "ascendc_kernel_cmake not found")
endif()
include("${ASCENDC_CMAKE_DIR}/ascendc.cmake")

set(INCLUDE_DIRS
    ${CMAKE_CURRENT_SOURCE_DIR}/include/kernel
    ${CMAKE_CURRENT_SOURCE_DIR}/include/block
    ${CMAKE_CURRENT_SOURCE_DIR}/include
)

ascendc_library(ascendc_kernels SHARED ${CMAKE_CURRENT_SOURCE_DIR}/src/matmul_all_gather.cpp)
ascendc_include_directories(ascendc_kernels PRIVATE ${INCLUDE_DIRS})

add_executable(test_matmul_all_gather ${CMAKE_CURRENT_SOURCE_DIR}/src/test_matmul_all_gather.cpp)
target_compile_options(test_matmul_all_gather PRIVATE -O2 -std=c++17 -D_GLIBCXX_USE_CXX11_ABI=0)
target_compile_definitions(test_matmul_all_gather PRIVATE SOC_VERSION="${SOC_VERSION}")
target_include_directories(test_matmul_all_gather PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/pkg_inc
    ${ASCEND_CANN_PACKAGE_PATH}/include
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include/adv_api
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include/tiling
    ${INCLUDE_DIRS}
)
target_link_libraries(test_matmul_all_gather PRIVATE
    ascendc_kernels
    tiling_api
    platform
    pthread
    hccl
    hcomm
)

创建build.sh编译脚本：

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/build.sh
#!/bin/bash
set -e

echo "Setting up environment..."
[ -n "$ASCEND_HOME_PATH" ] || { echo "ERROR: ASCEND_HOME_PATH not set"; exit 1; }
source "$ASCEND_HOME_PATH/set_env.sh"

SCRIPT_DIR=$(cd $(dirname ${BASH_SOURCE[0]}) && pwd)
cd $SCRIPT_DIR

echo "Building in: $SCRIPT_DIR/build"
rm -rf build
mkdir -p build
cd build

echo "Running CMake..."
cmake ..

echo "Compiling..."
make -j$(nproc)

if [ -f "lib/libascendc_kernels.so" ] && [ -f "test_matmul_all_gather" ]; then
    echo ""
    echo "Build successful!"
    echo "  Kernel library: $(pwd)/lib/libascendc_kernels.so"
    echo "  Test binary:    $(pwd)/test_matmul_all_gather"
else
    echo "Build failed!"
    exit 1
fi

---
### **Tiling结构体定义**

定义MatmulAllGather算子的Tiling数据结构。

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/include/tiling_data.h
#ifndef TILING_DATA_H
#define TILING_DATA_H

#include <stdint.h>
#include <kernel_tiling/kernel_tiling.h>

namespace MatmulAllGatherImpl {

constexpr static int64_t MAX_RANK_NUM = 64;

struct HcclA5OpResParam {
    uint64_t workSpace;
    uint64_t workSpaceSize;
    uint32_t rankId;
    uint32_t rankDim;
    uint64_t winSize;
    uint64_t windowsIn[MAX_RANK_NUM];
    uint64_t windowsOut[MAX_RANK_NUM];
    uint64_t xnAddr;
    uint64_t ckeAddr;
    uint64_t msAddr;
    uint64_t msSize;
};

struct MatmulAllGatherTilingInfo {
    uint64_t mDim;
    uint64_t kDim;
    uint64_t nDim;
    uint64_t aivNum;
    uint64_t totalWinSize;
    uint64_t baseM;
    uint64_t baseK;
    uint64_t baseN;
};

struct MatmulAllGatherTilingData {
    Mc2InitTiling mc2InitTiling;
    Mc2CcTiling mc2CcTiling;
    MatmulAllGatherTilingInfo tilingInfo;
};

}

#endif

---
### **工具函数定义**

提供Matmul计算所需的工具函数和常量定义。

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/include/utils.h
#ifndef UTILS_H
#define UTILS_H

#include "tiling_data.h"
#include "kernel_operator.h"

namespace MatmulAllGatherImpl {

using namespace AscendC;

constexpr static int64_t L0A_SIZE = 64 * 1024;
constexpr static int64_t L0B_SIZE = 64 * 1024;
constexpr static int64_t L0C_SIZE = 256 * 1024;
constexpr static int64_t L1_SIZE = 512 * 1024;

constexpr static uint32_t CUBE_BASE_M = 16;
constexpr static uint32_t CUBE_BASE_K = 16;
constexpr static uint32_t CUBE_BASE_N = 16;

__aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b)
{
    if (b == 0) return 0;
    return (a + b - 1) / b;
}

__aicore__ inline uint64_t FloorAlign(uint64_t a, uint64_t b)
{
    if (b == 0) return a;
    return (a / b) * b;
}

__aicore__ inline uint64_t Min(uint64_t a, uint64_t b)
{
    return (a < b) ? a : b;
}

template<AscendC::HardEvent event>
__aicore__ inline void SyncFunc()
{
    AscendC::TEventID eventID = GetTPipePtr()->FetchEventID<event>();
    AscendC::SetFlag<event>(eventID);
    AscendC::WaitFlag<event>(eventID);
}

}

#endif

---
### **MTE通信类实现**

实现AllGather通信的核心类，包含HCCL上下文初始化、状态同步、数据拷贝等功能。

提示：
1. InitHcclContextByAddr：从mc2Context初始化HCCL参数
2. WriteStatusToWin/ReadStatus：状态同步机制
3. CopyDataFromWin：AllGather核心实现
4. InitBuffer：初始化UB缓冲区

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/include/kernel/mte_comm.h
#ifndef MTE_COMM_H
#define MTE_COMM_H

#include "utils.h"
#include "kernel_operator.h"

namespace MatmulAllGatherImpl {

using namespace AscendC;

constexpr static uint32_t UB_ALIGN_BYTES = 32U;
constexpr static uint32_t FLOAT_ALIGN_NUM = 8U;
constexpr static uint32_t BUFFER_NUM = 2U;
constexpr static uint64_t STATE_WIN_SIZE = 1024UL * 1024UL;
constexpr static uint64_t SINGLE_STATE_REGION_SIZE = 462UL * 1024UL;
constexpr static uint32_t ZERONE_STATE_POS = 0U;
constexpr static uint64_t WIN_ADDR_ALIGN = 512UL;

template<typename DataType>
class MTECommunication {
public:
    __aicore__ inline MTECommunication() {};
    __aicore__ inline void InitHcclContextByAddr(GM_ADDR mc2Context);
    __aicore__ inline void InitGMTensor(GM_ADDR y, uint64_t winSpaceGmSize);
    __aicore__ inline void InitBuffer(TPipe *tPipe);
    __aicore__ inline void WriteStatusToWin();
    __aicore__ inline void ReadStatus();
    __aicore__ inline void CopyDataFromWin(uint64_t perRankM, uint64_t nDim);
    __aicore__ inline GM_ADDR GetWinAddrGm(uint32_t rankId, uint64_t offset = 0);
    __aicore__ inline GM_ADDR GetWinDataAddrGm(uint32_t rankId, uint32_t winFlag);
    __aicore__ inline GM_ADDR GetWinStatusAddrGm(uint32_t rankId, uint32_t winFlag);

    __gm__ HcclA5OpResParam *hcclContext_;
    uint32_t aivId_{0};
    uint32_t winBufferFlags_{0};
    GlobalTensor<DataType> outputTensor_;
    GlobalTensor<DataType> localWinOutTensor_;
    GM_ADDR localWinDataAddr_;

private:
    __aicore__ inline void CopyDataBlockFromWin(uint32_t remoteRankId, uint64_t readOffset, uint64_t curOutOffset, uint32_t count);

    TQueBind<QuePosition::VECIN, QuePosition::VECOUT, 1> copyQueue_;
    TBuf<> winFlagsBuf_;
    TBuf<> writeStateBuf_;
    TBuf<> readStateBuf_;
    TBuf<> stateResetBuf_;

    GlobalTensor<uint32_t> selfWinFlagGMTensor_;
    LocalTensor<DataType> copyTmpTensor_;
    LocalTensor<float> stateResetTensor_;
};

// TODO: 实现InitHcclContextByAddr
// TODO: 实现InitGMTensor
// TODO: 实现InitBuffer
// TODO: 实现WriteStatusToWin
// TODO: 实现ReadStatus
// TODO: 实现CopyDataFromWin

}

#endif

---
### **MatmulAllGather算子类实现**

实现MatmulAllGather算子类，包含Init、Process、ExecuteMatmul等方法。

提示：
1. Init函数：初始化Tiling参数、HCCL上下文、Matmul Block
2. Process函数：执行Matmul计算并调用AllGather
3. ExecuteMatmulToWin：Matmul结果写入Win Area（核心优化点）

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/include/kernel/matmul_all_gather.h
#ifndef MATMUL_ALL_GATHER_KERNEL_H
#define MATMUL_ALL_GATHER_KERNEL_H

#include "utils.h"
#include "mte_comm.h"
#include "block/simple_block_scheduler.h"
#include "block/simple_block_mmad.h"

namespace MatmulAllGatherImpl {

using namespace AscendC;

template<typename DataType>
class MatmulAllGatherOp {
public:
    using AType = DataType;
    using BType = DataType;
    using CType = DataType;
    using BlockMmadType = SimpleBlockMmad<AType, BType, CType>;

    __aicore__ inline MatmulAllGatherOp() {};
    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR y, GM_ADDR mc2Context, GM_ADDR tilingGM, TPipe *tPipe);
    __aicore__ inline void Process();

private:
    uint64_t mDim_{0};
    uint64_t kDim_{0};
    uint64_t nDim_{0};
    uint64_t totalWinSize_{0};
    int64_t baseM_{128};
    int64_t baseK_{128};
    int64_t baseN_{128};

    GM_ADDR aGm_;
    GM_ADDR bGm_;
    GM_ADDR cGm_;

    __gm__ MatmulAllGatherTilingData* tilingDataOut_;
    MTECommunication<DataType> mteComm_;

    BlockMmadType mmadOp_;
};

// TODO: 实现Init函数
// 提示：
// 1. 从tilingGM获取tilingDataOut_
// 2. 解析tilingInfo参数
// 3. 初始化mteComm_和mmadOp_

// TODO: 实现Process函数
// 提示：
// 1. 使用SimpleBlockScheduler切分M/N轴
// 2. 调用SimpleBlockMmad执行Matmul
// 3. Matmul结果写入GM（或Win Area）
// 4. AIV核执行AllGather通信

}

#endif

---
### **Kernel入口实现**

修改代码，完成Kernel侧核函数入口实现。

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/src/matmul_all_gather.cpp
#include "kernel_operator.h"
#include "kernel/matmul_all_gather.h"

using namespace AscendC;
using namespace MatmulAllGatherImpl;

// TODO: 实现MatmulAllGather_Generic核函数
// 提示：
// 1. 设置KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_2)
// 2. 创建TPipe对象
// 3. 创建MatmulAllGatherOp<bfloat16_t>对象
// 4. 调用op.Init和op.Process

extern "C" void matmul_all_gather_demo(uint32_t blockDim, void* stream,
    uint8_t* a, uint8_t* b, uint8_t* y, uint8_t* workspaceGM, uint8_t* mc2Context, uint8_t* tilingGM)
{
    // TODO: 调用MatmulAllGather_Generic<<<blockDim, nullptr, stream>>>
}

extern "C" {
    int HcomGetCommHandleByGroup(const char* groupName, void** commHandle);
    int HcclAllocComResourceByTiling(void* comm, void* stream, void* mc2Tiling, void** commContext);
}

---
### **Host侧主程序实现**

修改代码，完成Tiling计算、HCCL通信上下文创建和多卡测试。

提示：
1. CreateTilingDataAndContext：计算HCCL Buffer大小并创建通信上下文
2. GenerateRandomBF16Data：生成测试数据
3. 多卡执行：为每个rank创建stream、context、hcclComm

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/src/test_matmul_all_gather.cpp
#include <iostream>
#include <vector>
#include <string>
#include <cstring>
#include <getopt.h>
#include <cmath>
#include <cstdlib>
#include "hccl/hccl.h"
#include "acl/acl.h"
#include "hccl/hcom.h"
#include "hccl/hccl_comm.h"
#include "securec.h"
#include "tiling/hccl/hccl_tiling.h"
#include "tiling_data.h"

using namespace std;
using namespace MatmulAllGatherImpl;

static constexpr size_t BF16_SIZE = 2;

// TODO: 定义全局变量：gRankId, gRankSize, gM, gK, gN, gOutputDir
// TODO: 定义extern函数声明：matmul_all_gather_demo, HcclAllocComResourceByTiling

// TODO: 实现GenerateRandomBF16Data函数
// TODO: 实现CreateTilingDataAndContext函数
// TODO: 实现main函数的多卡测试逻辑

---
### **测试脚本 - 结果验证**

验证AllGather输出结果是否正确。

验证逻辑：
1. 对每个rank，计算Matmul结果
2. 模拟AllGather：将所有rank的C按顺序排列
3. 对比NPU输出与CPU预期结果（rankDim × M × N）

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/scripts/verify_precision.py
import numpy as np
import sys
import os

def verify_result(rank_id, rank_size, m, k, n, output_dir):
    # 读取NPU输出 (rankDim × M × N)
    npu_file = f'{output_dir}/output_rank{rank_id}.bin'
    if not os.path.exists(npu_file):
        print(f'[ERROR] NPU output not found: {npu_file}')
        return False
    
    npu_output = np.fromfile(npu_file, dtype=np.float16).reshape(rank_size * m, n)
    print(f'NPU output shape: {npu_output.shape}')
    
    # CPU计算：对每个rank计算Matmul并汇聚
    cpu_results = []
    for r in range(rank_size):
        a_file = f'{output_dir}/input_a_rank{r}.bin'
        b_file = f'{output_dir}/input_b.bin'
        
        if not os.path.exists(a_file) or not os.path.exists(b_file):
            print(f'[ERROR] Input files not found for rank {r}')
            return False
        
        a = np.fromfile(a_file, dtype=np.float16).reshape(m, k)
        b = np.fromfile(b_file, dtype=np.float16).reshape(k, n)
        
        # CPU Matmul
        c = np.matmul(a.astype(np.float32), b.astype(np.float32))
        cpu_results.append(c.astype(np.float16))
    
    # 模拟AllGather：按rank顺序拼接
    cpu_allgather = np.concatenate(cpu_results, axis=0)
    print(f'CPU AllGather shape: {cpu_allgather.shape}')
    
    # 对比结果
    diff = np.abs(cpu_allgather - npu_output)
    max_diff = np.max(diff)
    mean_diff = np.mean(diff)
    
    print(f'Max difference: {max_diff:.6f}')
    print(f'Mean difference: {mean_diff:.6f}')
    
    if max_diff > 0.1:
        print('[ERROR] Verification failed!')
        return False
    else:
        print('[PASS] Verification passed!')
        return True

if __name__ == '__main__':
    if len(sys.argv) != 7:
        print('Usage: python verify_precision.py <rank_id> <rank_size> <m> <k> <n> <output_dir>')
        sys.exit(1)
    
    rank_id = int(sys.argv[1])
    rank_size = int(sys.argv[2])
    m = int(sys.argv[3])
    k = int(sys.argv[4])
    n = int(sys.argv[5])
    output_dir = sys.argv[6]
    
    if verify_result(rank_id, rank_size, m, k, n, output_dir):
        print('\n========== Verification PASSED ==========')
        sys.exit(0)
    else:
        print('\n========== Verification FAILED ==========')
        sys.exit(1)

---
### **多卡测试脚本**

创建run_multi_data_test.sh脚本，用于自动化执行多卡测试。

In [ ]:
%%writefile Sources/01.04/matmul_all_gather/run_multi_data_test.sh
#!/bin/bash

BUILD_DIR=$(cd $(dirname ${BASH_SOURCE[0]}) && pwd)/build

if [ -z $ASCEND_HOME_PATH ]; then
    echo "ERROR: ASCEND_HOME_PATH not set"
    exit 1
fi

export LD_LIBRARY_PATH=${BUILD_DIR}/lib:${ASCEND_HOME_PATH}/lib64:${LD_LIBRARY_PATH}

EXECUTABLE=${BUILD_DIR}/test_matmul_all_gather
SCRIPTS_DIR=$(cd $(dirname ${BASH_SOURCE[0]}) && pwd)/scripts

RANK_SIZE=2
M=1024
K=1024
N=1024

WORK_DIR=${BUILD_DIR}/work_m${M}_n${N}_k${K}
mkdir -p ${WORK_DIR}

echo "=========================================="
echo "Test: M=${M}, N=${N}, K=${K}, RankSize=${RANK_SIZE}
echo "=========================================="

# 生成测试数据
echo "Generating test data..."
for rank in $(seq 0 $(($RANK_SIZE-1))); do
    python3 -c "
import numpy as np
np.random.seed(42 + $rank)
a = np.random.randn($M, $K).astype(np.float16)
a.tofile('${WORK_DIR}/input_a_rank${rank}.bin')
print('Generated input_a_rank${rank}.bin')
"
done

python3 -c "
import numpy as np
np.random.seed(100)
b = np.random.randn($K, $N).astype(np.float16)
b.tofile('${WORK_DIR}/input_b.bin')
print('Generated input_b.bin')
"

# 执行测试
echo "Running MatmulAllGather test..."
for rank in $(seq 0 $(($RANK_SIZE-1))); do
    ${EXECUTABLE} --rank_id=$rank --rank_size=$RANK_SIZE --m=$M --k=$K --n=$N --output_dir=${WORK_DIR} &
done
wait

# 验证结果
echo "Verifying results..."
python3 ${SCRIPTS_DIR}/verify_precision.py 0 $RANK_SIZE $M $K $N ${WORK_DIR}

if [ $? -eq 0 ]; then
    echo "========== Test PASSED =========="
    exit 0
else
    echo "========== Test FAILED =========="
    exit 1
fi

---
### **编译部署**

修改完成后，编译并部署算子进行测试。

In [ ]:
# 编译算子
!cd Sources/01.04/matmul_all_gather; bash build.sh

In [ ]:
# 执行测试
!cd Sources/01.04/matmul_all_gather; bash run_multi_data_test.sh

测试运行成功后会有类似以下输出：

```
========================================== 
Test: M=1024, N=1024, K=1024, RankSize=2
========================================== 

NPU output shape: (2048, 1024)
CPU AllGather shape: (2048, 1024)
Max difference: 0.002344
Mean difference: 0.000156
[PASS] Verification passed!

========== Test PASSED ========== 
```

---
#### **查看答案**

实现方式较为复杂，这里给出的答案仅供参考。由于代码较长，建议参考完整实现文件。

关键实现要点：

1. **Host侧Tiling计算**：
   - 动态计算HCCL Buffer大小：gHcclBufferSize = winDataSizeKB + 512
   - 使用Mc2CcTilingConfig配置AllGather通信参数
   - 调用HcclAllocComResourceByTiling创建通信上下文

2. **Kernel侧Matmul实现**：
   - 使用SimpleBlockScheduler切分M/N轴
   - 使用SimpleBlockMmad执行Matmul计算
   - Matmul结果写入GM（可优化为直接写入Win Area）

3. **AllGather实现**：
   - WriteStatusToWin：通知所有rank数据就绪
   - ReadStatus：等待所有rank数据就绪
   - CopyDataFromWin：从Win Area汇聚所有rank的C到输出GM

**参考答案路径**：
```
answer/01.04_answer/
├── include/
│   ├── tiling_data.h
│   ├── utils.h
│   ├── kernel/
│   │   ├── matmul_all_gather.h
│   │   └── mte_comm.h
│   └── block/
│       ├── simple_block_mmad.h
│       ├── simple_block_scheduler.h
│       └── block_scheduler_utils.h
├── src/
│   ├── matmul_all_gather.cpp
│   └── test_matmul_all_gather.cpp
```

In [ ]:
# 查看Kernel入口完整实现
!cat answer/01.04_answer/src/matmul_all_gather.cpp

In [ ]:
# 查看MatmulAllGatherOp核心实现
!head -n 80 answer/01.04_answer/include/kernel/matmul_all_gather.h

In [ ]:
# 查看MTECommunication通信实现
!head -n 100 answer/01.04_answer/include/kernel/mte_comm.h

In [ ]:
# 查看Host侧主程序
!head -n 150 answer/01.04_answer/src/test_matmul_all_gather.cpp

## 实践总结

通过本次实践，您已经掌握了MC2融合算子MatmulAllGather的完整开发流程：

1. **算子工程创建**：使用CMake工程结构，适合多卡分布式场景

2. **Host侧开发**：
   - Tiling参数计算（M/K/N、baseM/baseK/baseN）
   - HCCL Buffer大小动态计算
   - HCCL通信上下文创建

3. **Kernel侧开发**：
   - SimpleBlockScheduler切分调度
   - SimpleBlockMmad Matmul计算
   - Win Area状态同步
   - AllGather数据汇聚

4. **测试验证**：
   - 多卡数据生成
   - CPU模拟AllGather验证
   - 结果对比分析

**关键知识点**：
- MC2融合算子将计算与通信融合，减少内存拷贝
- AllGather输出大小 = Matmul输出 × rankDim（汇聚）
- SimpleBlockScheduler相比ReduceScatter的复杂调度更简单
- 状态同步机制：确保多rank数据就绪后再执行AllGather

**与ReduceScatter对比**：
- **ReduceScatter**：切分+Reduce，输出大小 = 输入 / rankDim
- **AllGather**：汇聚，输出大小 = 输入 × rankDim
- **通信复杂度**：ReduceScatter需要ReduceSum，AllGather仅拷贝

**扩展练习**：
- 尝试优化Matmul结果直接写入Win Area（避免GM中间拷贝）
- 扩展支持更大的M/N/K维度（如2048、4096）
- 对比不同rankSize（2卡 vs 4卡）的性能差异